In [ ]:
# ═══════════════════════════════════════════════════════
# 0. 환경 설정
# ═══════════════════════════════════════════════════════
!pip install lightgbm xgboost torch tqdm --quiet

import os, random, re, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgbm
from tqdm.auto import tqdm

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/비타민/풍력발전량예측/data")  # FIX: 기존 경로에 실제 데이터 없음 (버그)
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}


In [ ]:
# ═══════════════════════════════════════════════════════
# 1. v10 데이터셋
# ═══════════════════════════════════════════════════════
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def dms_to_decimal(s):
    pattern = r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])"
    m = re.match(pattern, s.strip())
    d1, m1, s1, hemi1, d2, m2, s2, hemi2 = m.groups()
    lat = float(d1) + float(m1) / 60 + float(s1) / 3600
    if hemi1 == "S": lat = -lat
    lon = float(d2) + float(m2) / 60 + float(s2) / 3600
    if hemi2 == "W": lon = -lon
    return lat, lon

def find_nearest_grid(weather_df, coord_df, name_col, lat_col="lat", lon_col="lon"):
    grids = weather_df[["grid_id", "latitude", "longitude"]].drop_duplicates()
    mapping = {}
    for _, row in coord_df.iterrows():
        dists = haversine(row[lat_col], row[lon_col], grids["latitude"], grids["longitude"])
        mapping[row[name_col]] = grids.loc[dists.idxmin(), "grid_id"]
    return mapping

def add_wind_features(df, uv_pairs):
    df = df.copy()
    for u_col, v_col, height in uv_pairs:
        df[f"ws_{height}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        wd_rad = np.arctan2(df[v_col], df[u_col])
        df[f"wd_sin_{height}"] = np.sin(wd_rad)
        df[f"wd_cos_{height}"] = np.cos(wd_rad)
    return df

def build_group_weather(weather_df, group_grid_map, prefix):
    weather_df = weather_df.copy()
    weather_df["forecast_kst_dtm"] = pd.to_datetime(weather_df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in weather_df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    frames = []
    for group, grid_id in group_grid_map.items():
        sub = weather_df.loc[weather_df["grid_id"] == grid_id, ["forecast_kst_dtm", *value_cols]].copy()
        sub = sub.rename(columns={c: f"{prefix}_{c}" for c in value_cols})
        sub["kpx_group"] = int(group)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

def add_forecast_disagreement(df):
    df = df.copy()
    df["ws_disagreement"] = np.abs(df["ldaps_ws_10m"] - df["gfs_ws_10m"])
    df["wd_agreement"] = df["ldaps_wd_cos_10m"] * df["gfs_wd_cos_10m"] + df["ldaps_wd_sin_10m"] * df["gfs_wd_sin_10m"]
    return df

def add_seasonal_interaction(df):
    df = df.copy()
    month = pd.to_datetime(df["forecast_kst_dtm"]).dt.month
    df["high_variance_season"] = month.isin([1, 2, 3, 4, 5, 7]).astype(int)
    df["ws_x_season"] = df["ldaps_ws_10m"] * df["high_variance_season"]
    df["disagreement_x_season"] = df["ws_disagreement"] * df["high_variance_season"]
    return df

def build_datasets():
    train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
    gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
    ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
    gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")
    train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
    sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

    raw = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=None)
    info = raw.iloc[4:21].copy()
    info.columns = raw.iloc[3]
    info = info.reset_index(drop=True)
    info = info.rename(columns={"좌표(Google)": "coord", "KPX그룹": "kpx_group", "설비용량(MW)": "capacity_mw"})
    info["kpx_group"] = info["kpx_group"].ffill().astype(int)
    coords = info["coord"].apply(dms_to_decimal)
    info["lat"] = coords.apply(lambda x: x[0])
    info["lon"] = coords.apply(lambda x: x[1])
    assert info.groupby("kpx_group")["capacity_mw"].sum().tolist() == [21.6, 21.6, 21.0]

    group_coords = info.groupby("kpx_group")[["lat", "lon"]].mean().reset_index()
    group_coords = group_coords.rename(columns={"kpx_group": "KPX그룹"})
    ldaps_group_grid = find_nearest_grid(ldaps_train, group_coords, "KPX그룹")
    gfs_group_grid = find_nearest_grid(gfs_train, group_coords, "KPX그룹")

    LDAPS_UV_PAIRS = [("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m")]
    GFS_UV_PAIRS = [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "80m"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "100m"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl"),
    ]
    ldaps_train = add_wind_features(ldaps_train, LDAPS_UV_PAIRS)
    ldaps_test = add_wind_features(ldaps_test, LDAPS_UV_PAIRS)
    gfs_train = add_wind_features(gfs_train, GFS_UV_PAIRS)
    gfs_test = add_wind_features(gfs_test, GFS_UV_PAIRS)
    for df in [gfs_train, gfs_test]:
        df["wind_shear_100_10"] = df["ws_100m"] - df["ws_10m"]
        df["wind_shear_80_10"] = df["ws_80m"] - df["ws_10m"]

    ldaps_group_train = build_group_weather(ldaps_train, ldaps_group_grid, "ldaps")
    ldaps_group_test = build_group_weather(ldaps_test, ldaps_group_grid, "ldaps")
    gfs_group_train = build_group_weather(gfs_train, gfs_group_grid, "gfs")
    gfs_group_test = build_group_weather(gfs_test, gfs_group_grid, "gfs")

    train_labels_long = train_labels.melt(id_vars=["kst_dtm"], value_vars=TARGET_COLS,
                                          var_name="kpx_group_col", value_name="target")
    train_labels_long["kpx_group"] = train_labels_long["kpx_group_col"].str.extract(r"(\d+)").astype(int)
    train_labels_long = train_labels_long.rename(columns={"kst_dtm": "forecast_kst_dtm"}).drop(columns=["kpx_group_col"])

    test_base = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    test_long = pd.concat([test_base.assign(kpx_group=g) for g in [1, 2, 3]], ignore_index=True)

    train_df = train_labels_long.merge(ldaps_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                                  .merge(gfs_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left")
    test_df = test_long.merge(ldaps_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                        .merge(gfs_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left")
    assert train_df.shape[0] == train_labels_long.shape[0]

    train_df = add_seasonal_interaction(add_forecast_disagreement(train_df))
    test_df = add_seasonal_interaction(add_forecast_disagreement(test_df))
    weather_cols = [c for c in train_df.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]
    print(f"v10 피처 개수: {len(weather_cols)}")
    return train_df, test_df, weather_cols, sample_submission, info

In [ ]:
# ═══════════════════════════════════════════════════════
# 2. 공통 유틸 — metric, 편향보정, fold
# ═══════════════════════════════════════════════════════
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"], out["day"], out["hour"] = dt.dt.month, dt.dt.day, dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24); out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24)
    out["month_sin"] = np.sin(2*np.pi*out["month"]/12); out["month_cos"] = np.cos(2*np.pi*out["month"]/12)
    return out

def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float); forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]; valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae); ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr

def make_sample_weight(y, capacity, low_weight=0.05):
    return np.where(y >= capacity * 0.10, 1.0, low_weight)

def apply_rolling_smoothing(pred_series, window=3, alpha=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    return alpha * pred_series + (1 - alpha) * rolling

def apply_bias_correction_v2(pred, ws, group, bias_map):
    pred = pred.copy()
    if group not in bias_map: return pred
    for interval, bias in bias_map[group].items():
        mask = (ws > interval.left) & (ws <= interval.right)
        pred[mask] = pred[mask] * (1 - bias)
    return pred

def lgbm_params(n_estimators, n_jobs, seed, objective="mae"):
    # GPU(OpenCL) 벤치마크: CPU 143s -> GPU 74s (60k행x352열, 500트리 기준)
    return dict(n_estimators=n_estimators, max_depth=6, learning_rate=0.03, num_leaves=31,
                min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=n_jobs, verbose=-1, objective=objective, device="gpu")

def prepare_matrices(tr_g, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_g["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_g[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

FOLDS_Q = [
    {"name": "fold_2024Q1", "train_end": pd.Timestamp("2024-01-01"), "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2024-04-01")},
    {"name": "fold_2024Q2", "train_end": pd.Timestamp("2024-04-01"), "valid_start": pd.Timestamp("2024-04-01 01:00:00"), "valid_end": pd.Timestamp("2024-07-01")},
    {"name": "fold_2024Q3", "train_end": pd.Timestamp("2024-07-01"), "valid_start": pd.Timestamp("2024-07-01 01:00:00"), "valid_end": pd.Timestamp("2024-10-01")},
    {"name": "fold_2024Q4", "train_end": pd.Timestamp("2024-10-01"), "valid_start": pd.Timestamp("2024-10-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")},
]
HOLD_FOLD = {"name": "hold_2024full", "train_end": pd.Timestamp("2024-01-01"),
             "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")}
FOLD_LIST = [(f["name"], f) for f in FOLDS_Q] + [(HOLD_FOLD["name"], HOLD_FOLD)]

COARSE_BINS = [0, 11, 13, 100]

def estimate_bias_data(train_df, weather_cols, tr_end, n_estimators, n_jobs,
                       low_weight=0.05, holdout_ratio=0.25, min_fit_days=90):
    out = {}
    for group in [1, 2, 3]:
        capacity = CAPACITY_KWH[f"kpx_group_{group}"]
        g_data = train_df[(train_df["kpx_group"] == group) & (train_df["forecast_kst_dtm"] <= tr_end)]
        g_data = g_data[g_data["target"].notna()]
        if len(g_data) == 0: continue
        first_date = g_data["forecast_kst_dtm"].min()
        available_days = max((tr_end - first_date).days, 1)
        holdout_days = min(int(available_days * holdout_ratio), 180)
        if available_days - holdout_days < min_fit_days:
            holdout_days = max(0, available_days - min_fit_days)
        tr_sub_end = tr_end - pd.Timedelta(days=holdout_days)
        fit_g = g_data[g_data["forecast_kst_dtm"] <= tr_sub_end]
        check_g = g_data[g_data["forecast_kst_dtm"] > tr_sub_end]
        if len(fit_g) == 0 or len(check_g) == 0: continue
        fit_feat = pd.concat([calendar_features(fit_g["forecast_kst_dtm"]).reset_index(drop=True),
                              fit_g[weather_cols].reset_index(drop=True)], axis=1)
        check_feat = pd.concat([calendar_features(check_g["forecast_kst_dtm"]).reset_index(drop=True),
                                check_g[weather_cols].reset_index(drop=True)], axis=1)
        imputer = SimpleImputer(strategy="median")
        X_fit = pd.DataFrame(imputer.fit_transform(fit_feat), columns=fit_feat.columns)
        X_check = pd.DataFrame(imputer.transform(check_feat), columns=check_feat.columns)
        w_fit = make_sample_weight(fit_g["target"].to_numpy(), capacity, low_weight)
        model = lgb.LGBMRegressor(**lgbm_params(n_estimators, n_jobs, SEED))
        model.fit(X_fit, fit_g["target"], sample_weight=w_fit)
        pred_check = model.predict(X_check)
        check_g = check_g.copy()
        check_g["signed_error_rate"] = (pred_check - check_g["target"]) / capacity
        check_g = check_g.loc[check_g["target"] >= capacity * 0.10]
        if len(check_g) == 0: continue
        out[group] = check_g[["ldaps_ws_10m", "signed_error_rate"]]
    return out

def bias_map_from_data(bias_data, bins_by_group):
    bias_map = {}
    for group, df in bias_data.items():
        ws_bin = pd.cut(df["ldaps_ws_10m"], bins=bins_by_group[group])
        bias_map[group] = df.groupby(ws_bin, observed=True)["signed_error_rate"].mean().to_dict()
    return bias_map

In [ ]:
# ═══════════════════════════════════════════════════════
# 3. v10TPS 피처 (T+P+S 블록)
# ═══════════════════════════════════════════════════════
def add_physics(df):
    df = df.copy()
    rho = df["ldaps_surface_0_sp"] / (287.05 * df["ldaps_heightAboveGround_2_t"])
    df["phy_density"] = rho
    df["phy_ws10_cu"] = df["ldaps_ws_10m"] ** 3
    df["phy_ws100_cu"] = df["gfs_ws_100m"] ** 3
    df["phy_rho_ws100_cu"] = rho * df["phy_ws100_cu"]
    df["phy_pc100"] = np.clip(df["gfs_ws_100m"], 0, 12) ** 3 / 12 ** 3
    ws50max = np.hypot(df["ldaps_heightAboveGround_50_50MUmax"], df["ldaps_heightAboveGround_50_50MVmax"])
    ws50min = np.hypot(df["ldaps_heightAboveGround_50_50MUmin"], df["ldaps_heightAboveGround_50_50MVmin"])
    df["phy_ws50max"] = ws50max
    df["phy_gust_range"] = ws50max - ws50min
    df["phy_gust_over_ws"] = ws50max / (df["ldaps_ws_10m"] + 0.5)
    df["phy_gfs_gust_factor"] = df["gfs_surface_0_gust"] / (df["gfs_ws_10m"] + 0.5)
    ws850 = np.hypot(df["gfs_isobaricInhPa_850_u"], df["gfs_isobaricInhPa_850_v"])
    df["phy_ws850"] = ws850
    df["phy_shear_850_100"] = ws850 - df["gfs_ws_100m"]
    cols = [c for c in df.columns if c.startswith("phy_")]
    return df, cols

def add_temporal(df):
    df = df.sort_values(["kpx_group", "forecast_kst_dtm"]).copy()
    for src in ["ldaps_ws_10m", "gfs_ws_100m"]:
        g = df.groupby("kpx_group")[src]
        p = "tmp_" + ("l10" if src.startswith("ldaps") else "g100")
        df[f"{p}_r3_mean"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
        df[f"{p}_r3_std"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).std())
        df[f"{p}_r7_mean"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).mean())
        df[f"{p}_r7_std"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).std())
        df[f"{p}_d1"] = g.transform(lambda s: s.diff(1))
        df[f"{p}_dm1"] = g.transform(lambda s: s.diff(-1))
        df[f"{p}_d3"] = g.transform(lambda s: s.diff(3))
        df[f"{p}_dm3"] = g.transform(lambda s: s.diff(-3))
    day_key = (pd.to_datetime(df["forecast_kst_dtm"]) - pd.Timedelta(hours=1)).dt.date
    day_grp = df.groupby([df["kpx_group"], day_key])["ldaps_ws_10m"]
    df["tmp_day_mean"] = day_grp.transform("mean")
    df["tmp_day_max"] = day_grp.transform("max")
    df["tmp_day_std"] = day_grp.transform("std")
    cols = [c for c in df.columns if c.startswith("tmp_")]
    return df, cols

def build_spatial_frames(info):
    centroids = info.groupby("kpx_group")[["lat", "lon"]].mean()
    out = {}
    for split, ld_path, gf_path in [("train", TRAIN_DIR / "ldaps_train.csv", TRAIN_DIR / "gfs_train.csv"),
                                    ("test", TEST_DIR / "ldaps_test.csv", TEST_DIR / "gfs_test.csv")]:
        ld = pd.read_csv(ld_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude",
                                  "heightAboveGround_10_10u", "heightAboveGround_10_10v"])
        ld["forecast_kst_dtm"] = pd.to_datetime(ld["forecast_kst_dtm"])
        ld["ws"] = np.hypot(ld["heightAboveGround_10_10u"], ld["heightAboveGround_10_10v"])
        grids = ld[["grid_id", "latitude", "longitude"]].drop_duplicates().set_index("grid_id")
        frames = []
        for group in [1, 2, 3]:
            d = haversine(centroids.loc[group, "lat"], centroids.loc[group, "lon"], grids["latitude"], grids["longitude"])
            near = d.nsmallest(4).index
            sub = ld[ld["grid_id"].isin(near)].groupby("forecast_kst_dtm")["ws"]
            agg = pd.DataFrame({"spa_l10_mean": sub.mean(), "spa_l10_std": sub.std(),
                                "spa_l10_range": sub.max() - sub.min()}).reset_index()
            agg["kpx_group"] = group
            frames.append(agg)
        ldaps_spa = pd.concat(frames, ignore_index=True)
        gf = pd.read_csv(gf_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "heightAboveGround_100_100u", "heightAboveGround_100_100v"])
        gf["forecast_kst_dtm"] = pd.to_datetime(gf["forecast_kst_dtm"])
        gf["ws100"] = np.hypot(gf["heightAboveGround_100_100u"], gf["heightAboveGround_100_100v"])
        sub = gf.groupby("forecast_kst_dtm")["ws100"]
        gfs_spa = pd.DataFrame({"spa_g100_mean": sub.mean(), "spa_g100_std": sub.std()}).reset_index()
        out[split] = (ldaps_spa, gfs_spa)
    return out

def build_v11(info):
    train_df, test_df, weather_cols_v10, sample_submission, _ = build_datasets()
    spa = build_spatial_frames(info)
    dfs = {}
    for split, df in [("train", train_df), ("test", test_df)]:
        df, phy_cols = add_physics(df)
        df, tmp_cols = add_temporal(df)
        ldaps_spa, gfs_spa = spa[split]
        df = df.merge(ldaps_spa, on=["forecast_kst_dtm", "kpx_group"], how="left")
        df = df.merge(gfs_spa, on="forecast_kst_dtm", how="left")
        dfs[split] = df
    spa_cols = ["spa_l10_mean", "spa_l10_std", "spa_l10_range", "spa_g100_mean", "spa_g100_std"]
    v10tps_cols = weather_cols_v10 + tmp_cols + phy_cols + spa_cols
    return dfs["train"], dfs["test"], v10tps_cols, sample_submission

In [ ]:
# ═══════════════════════════════════════════════════════
# 4. spd 다격자 편차 피처
# ═══════════════════════════════════════════════════════
SPD_PATH = CACHE_DIR / "spatial_spd.parquet"
LD_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "u10": lambda d: d["heightAboveGround_10_10u"], "v10": lambda d: d["heightAboveGround_10_10v"],
    "ws50x": lambda d: np.hypot(d["heightAboveGround_50_50MUmax"], d["heightAboveGround_50_50MVmax"]),
    "wsbl": lambda d: np.hypot(d["heightAboveGround_5_XBLWS"], d["heightAboveGround_5_YBLWS"]),
    "t2": lambda d: d["heightAboveGround_2_t"], "sp": lambda d: d["surface_0_sp"], "blh": lambda d: d["etc_0_blh"],
}
LD_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_50_50MUmax",
          "heightAboveGround_50_50MVmax", "heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS",
          "heightAboveGround_2_t", "surface_0_sp", "etc_0_blh"]
GF_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "ws100": lambda d: np.hypot(d["heightAboveGround_100_100u"], d["heightAboveGround_100_100v"]),
    "u100": lambda d: d["heightAboveGround_100_100u"], "v100": lambda d: d["heightAboveGround_100_100v"],
    "gust": lambda d: d["surface_0_gust"],
    "ws850": lambda d: np.hypot(d["isobaricInhPa_850_u"], d["isobaricInhPa_850_v"]),
    "t850": lambda d: d["isobaricInhPa_850_t"], "prmsl": lambda d: d["meanSea_0_prmsl"],
}
GF_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_100_100u",
          "heightAboveGround_100_100v", "surface_0_gust", "isobaricInhPa_850_u",
          "isobaricInhPa_850_v", "isobaricInhPa_850_t", "meanSea_0_prmsl"]

def _load(paths, src_cols):
    frames = []
    for p in paths:
        d = pd.read_csv(p, encoding="utf-8-sig", usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude", *src_cols])
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def _pivot(df, fn, master_idx, gid):
    v = df[["forecast_kst_dtm", "grid_id"]].copy()
    v["val"] = np.asarray(fn(df), dtype=np.float64)
    M = v.pivot_table(index="forecast_kst_dtm", columns="grid_id", values="val", aggfunc="mean")
    M = M.reindex(index=master_idx, columns=gid)
    row_mean = M.mean(axis=1)
    return M.apply(lambda c: c.fillna(row_mean))

def build_spatial(prefix, paths, src_cols, var_defs):
    df = _load(paths, src_cols)
    grids = df[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id")
    gid = grids["grid_id"].to_numpy()
    master_idx = pd.Index(np.sort(df["forecast_kst_dtm"].unique()), name="forecast_kst_dtm")
    spd_cols = {}
    for name, fn in var_defs.items():
        M = _pivot(df, fn, master_idx, gid)
        V = M.to_numpy()
        mean = V.mean(axis=1)
        spd_cols[f"sd{prefix}_{name}_mean"] = mean
        for j, g in enumerate(gid):
            spd_cols[f"sd{prefix}_{name}_g{int(g)}"] = V[:, j] - mean
    return pd.DataFrame(spd_cols, index=master_idx).reset_index()

def build_spd(force=False):
    if not force and SPD_PATH.exists():
        spd = pd.read_parquet(SPD_PATH)
    else:
        t0 = time.time()
        ld_spd = build_spatial("L", [TRAIN_DIR / "ldaps_train.csv", TEST_DIR / "ldaps_test.csv"], LD_SRC, LD_VARS)
        gf_spd = build_spatial("G", [TRAIN_DIR / "gfs_train.csv", TEST_DIR / "gfs_test.csv"], GF_SRC, GF_VARS)
        spd = ld_spd.merge(gf_spd, on="forecast_kst_dtm", how="outer")
        spd.to_parquet(SPD_PATH, index=False)
        print(f"spd 캐시 생성 완료 ({time.time()-t0:.0f}s)")
    spd_cols = [c for c in spd.columns if c != "forecast_kst_dtm"]
    return spd, spd_cols

In [ ]:
# ═══════════════════════════════════════════════════════
# 5. JMA 외부예보 수집
# ═══════════════════════════════════════════════════════
import requests

JMA_PATH = DATA_DIR / "external"
JMA_PATH.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
MODELS = {"jma_gsm": ["2022", "2023", "2024", "2025"], "jma_msm": ["2023", "2024", "2025"]}
HOURLY_VARS = ["wind_speed_10m", "wind_direction_10m", "temperature_2m", "surface_pressure"]
LEADS = ["previous_day2", "previous_day1"]

def fetch_json(url, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            d = r.json()
            if "hourly" in d:
                return d
            if "reason" in d:
                raise RuntimeError(f"API error: {d['reason']}")
        except Exception as e:
            print(f"  재시도 {attempt+1}: {e}")
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"fetch 실패: {url[:120]}")

def get_centroids(info):
    return info.groupby("kpx_group")[["lat", "lon"]].mean()

def fetch_jma_prevruns(info, force=False):
    out_path = JMA_PATH / "jma_prevruns.csv"
    if not force and out_path.exists():
        return pd.read_csv(out_path, encoding="utf-8-sig", parse_dates=["forecast_kst_dtm"])
    centroids = get_centroids(info)
    hourly_param = ",".join(f"{v}_{lead}" for v in HOURLY_VARS for lead in LEADS)
    frames = []
    for group in [1, 2, 3]:
        lat = round(float(centroids.loc[group, "lat"]), 4)
        lon = round(float(centroids.loc[group, "lon"]), 4)
        for model, years in MODELS.items():
            for year in years:
                url = (f"{BASE_URL}?latitude={lat}&longitude={lon}&models={model}"
                       f"&hourly={hourly_param}&wind_speed_unit=ms"
                       f"&start_date={year}-01-01&end_date={year}-12-31&timezone=Asia%2FSeoul")
                d = fetch_json(url)
                h = d["hourly"]
                df = pd.DataFrame({"forecast_kst_dtm": pd.to_datetime(h["time"])})
                for v in HOURLY_VARS:
                    for lead in LEADS:
                        short = {"previous_day2": "d2", "previous_day1": "d1"}[lead]
                        df[f"{v}_{short}"] = h.get(f"{v}_{lead}")
                df["kpx_group"] = group
                df["model"] = model
                frames.append(df)
                time.sleep(1)
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {out_path} ({len(out)}행)")
    return out

def build_jma_frame(jma_df):
    frames = []
    for model, tag in [("jma_gsm", "jgsm"), ("jma_msm", "jmsm")]:
        sub = jma_df[jma_df["model"] == model].copy()
        wd_rad = np.deg2rad(sub["wind_direction_10m_d2"])
        out = pd.DataFrame({
            "forecast_kst_dtm": sub["forecast_kst_dtm"], "kpx_group": sub["kpx_group"],
            f"{tag}_ws": sub["wind_speed_10m_d2"], f"{tag}_wdsin": np.sin(wd_rad), f"{tag}_wdcos": np.cos(wd_rad),
        })
        if model == "jma_gsm":
            out[f"{tag}_t2m"] = sub["temperature_2m_d2"]
            out[f"{tag}_sp"] = sub["surface_pressure_d2"]
        frames.append(out)
    return frames[0].merge(frames[1], on=["forecast_kst_dtm", "kpx_group"], how="outer")

JG_COLS = ["jgsm_ws", "jgsm_wdsin", "jgsm_wdcos", "jgsm_t2m", "jgsm_sp", "jgsm_dis"]
JM_COLS = ["jmsm_ws", "jmsm_wdsin", "jmsm_wdcos", "jmsm_dis"]

In [ ]:
# ═══════════════════════════════════════════════════════
# 5b. ECMWF IFS HRES 외부예보 수집 (open-data S3 아카이브, D-2 12z oper 런)
#   wonwoo 브랜치 exp054_fetch_ecmwf.py / exp054_gate.py 이식.
#   - 규정: D-2일 12z 런 사용 → 예측기준시점(전날 13시 KST) 이전 발행 완료.
#   - Windows에서는 eccodes DLL이 앱 제어 정책에 막혀 fetch/decode를 분리했었지만,
#     이 환경(Linux)에서는 eccodes가 pip만으로 바로 동작해 한 번에 처리한다.
#   - GBM 전용 피처 4개: ec_ws, ec_wdsin, ec_wdcos, ec_dis (DL은 안 받음 — exp-040 교훈:
#     같은 정보를 GBM·DL 양쪽에 주면 오히려 탈상관이 죽어 상쇄된다).
# ═══════════════════════════════════════════════════════
import json
import eccodes as _eccodes
from concurrent.futures import ThreadPoolExecutor, as_completed

EC_PATH = DATA_DIR / "external"
EC_PATH.mkdir(parents=True, exist_ok=True)
EC_RAW_PARQUET = EC_PATH / "ecmwf_prevruns.parquet"
EC_S3 = "https://ecmwf-forecasts.s3.eu-central-1.amazonaws.com"
EC_STEPS = list(range(27, 52, 3))          # 27~51h, 3시간 간격 9개
EC_START, EC_END = "2023-01-18", "2025-12-30"   # 커버리지 시작(아카이브 실측) ~ 테스트기간 D-2 끝

_ec_sess = requests.Session()
_ec_sess.headers["User-Agent"] = "baram2026-simplified"

def _ec_http_get(url, rng=None, retries=6):
    # S3 아카이브가 간헐적으로 503(SlowDown/일시 과부하)을 반환함 — 소규모 스모크 테스트에서
    # 동일 URL이 재시도 시 바로 200으로 성공하는 걸 확인, 재시도 강화로 대응 (지수 백오프, 최대 20초).
    for a in range(retries):
        try:
            h = {"Range": f"bytes={rng[0]}-{rng[0]+rng[1]-1}"} if rng else {}
            r = _ec_sess.get(url, headers=h, timeout=60)
            if r.status_code in (200, 206):
                return r.content
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(min(2 * (2 ** a), 20))
    return None

def _ec_bilinear(msg_bytes, pts):
    """GRIB 메시지에서 각 (lat, lon) 지점의 이중선형 보간값"""
    gid = _eccodes.codes_new_from_message(msg_bytes)
    try:
        ni = _eccodes.codes_get(gid, "Ni"); nj = _eccodes.codes_get(gid, "Nj")
        lat0 = _eccodes.codes_get(gid, "latitudeOfFirstGridPointInDegrees")
        lon0 = _eccodes.codes_get(gid, "longitudeOfFirstGridPointInDegrees")
        dlat = _eccodes.codes_get(gid, "jDirectionIncrementInDegrees")
        dlon = _eccodes.codes_get(gid, "iDirectionIncrementInDegrees")
        jpos = _eccodes.codes_get(gid, "jScansPositively")
        vals = _eccodes.codes_get_values(gid).reshape(nj, ni)
    finally:
        _eccodes.codes_release(gid)
    slat = dlat if jpos else -dlat
    out = []
    for lat, lon in pts:
        fi = ((lon - lon0) % 360.0) / dlon
        fj = (lat - lat0) / slat
        i0, j0 = int(np.floor(fi)), int(np.floor(fj))
        wi, wj = fi - i0, fj - j0
        j0 = min(max(j0, 0), nj - 2)
        v = ((1 - wi) * (1 - wj) * vals[j0, i0 % ni] + wi * (1 - wj) * vals[j0, (i0 + 1) % ni]
             + (1 - wi) * wj * vals[j0 + 1, i0 % ni] + wi * wj * vals[j0 + 1, (i0 + 1) % ni])
        out.append(float(v))
    return out

def _ec_fetch_day(init_d, pts):
    d = init_d.strftime("%Y%m%d")
    roots = (["12z/0p4-beta/oper", "12z/ifs/0p4-beta/oper", "12z/ifs/0p25/oper"]
             if init_d < pd.Timestamp("2024-02-01") else
             ["12z/ifs/0p25/oper", "12z/ifs/0p4-beta/oper", "12z/0p4-beta/oper"])
    root, idx0 = None, None
    for cand in roots:
        idx0 = _ec_http_get(f"{EC_S3}/{d}/{cand}/{d}120000-27h-oper-fc.index", retries=5)
        if idx0:
            root = cand; break
    if root is None:
        return []
    rows = []
    for s in EC_STEPS:
        raw = idx0 if s == 27 else _ec_http_get(f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.index")
        if raw is None:
            continue
        ranges = {}
        for line in raw.decode("utf-8", "ignore").splitlines():
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            if e.get("levtype") == "sfc" and e.get("param") in ("10u", "10v"):
                ranges[e["param"]] = (int(e["_offset"]), int(e["_length"]))
        if "10u" not in ranges or "10v" not in ranges:
            continue
        gurl = f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.grib2"
        mu = _ec_http_get(gurl, rng=ranges["10u"])
        mv = _ec_http_get(gurl, rng=ranges["10v"])
        if mu is None or mv is None:
            continue
        us = _ec_bilinear(mu, pts)
        vs = _ec_bilinear(mv, pts)
        for g, u, v in zip([1, 2, 3], us, vs):
            rows.append(dict(init_date=d, step=s, group=g, u10=u, v10=v))
    return rows

def fetch_ecmwf_prevruns(info, start=EC_START, end=EC_END, workers=20, force=False):
    # 체크포인트: 실행 환경이 중간에 죽어도(관측됨) 이어서 진행하도록 100일마다 부분 저장.
    inits = pd.date_range(start, end, freq="D")
    all_rows, done_inits = [], set()
    if not force and EC_RAW_PARQUET.exists():
        prev = pd.read_parquet(EC_RAW_PARQUET)
        done_inits = set(prev["init_date"].unique())
        all_rows = prev.to_dict("records")
        if done_inits >= set(d.strftime("%Y%m%d") for d in inits):
            print(f"ECMWF raw 캐시 로드: {len(prev)}행, {len(done_inits)}일 (완료)")
            return prev
        print(f"ECMWF 부분 캐시 로드: {len(prev)}행, {len(done_inits)}일 — 이어서 진행", flush=True)
    todo = [d for d in inits if d.strftime("%Y%m%d") not in done_inits]
    centroids = get_centroids(info)
    pts = [(float(centroids.loc[g, "lat"]), float(centroids.loc[g, "lon"])) for g in [1, 2, 3]]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(_ec_fetch_day, d, pts): d for d in todo}
        n_done, n_ok = 0, 0
        for fut in as_completed(futs):
            r = fut.result()
            n_done += 1
            if r:
                n_ok += 1
                all_rows.extend(r)
            if n_done % 100 == 0:
                el = time.time() - t0
                print(f"  진행 {n_done}/{len(todo)}일 ({el/60:.1f}분, 성공 {n_ok}일, {el/n_done*len(todo)/60:.0f}분 예상)", flush=True)
                pd.DataFrame(all_rows).to_parquet(EC_RAW_PARQUET, index=False)
    out = pd.DataFrame(all_rows)
    out.to_parquet(EC_RAW_PARQUET, index=False)
    print(f"ECMWF raw 수집 완료: {len(out)}행, {out['init_date'].nunique()}/{len(inits)}일 ({(time.time()-t0)/60:.1f}분)", flush=True)
    return out

def build_ec_frame(ec_raw):
    """(init_date, step, group, u10, v10) → 시간별 KST ec_ws/ec_wdsin/ec_wdcos (exp054_gate.py build_ec_frame 이식)"""
    ec_raw = ec_raw.copy()
    ec_raw["init_date"] = pd.to_datetime(ec_raw["init_date"])
    frames = []
    for (init_d, g), sub in ec_raw.groupby(["init_date", "group"]):
        sub = sub.sort_values("step")
        if len(sub) < 2:
            continue
        anchor_h = sub["step"].to_numpy(dtype=float)             # 27~51
        target_h = np.arange(28, 52, dtype=float)                # D 01:00 ~ D+1 00:00 (24개)
        u = np.interp(target_h, anchor_h, sub["u10"].to_numpy())
        v = np.interp(target_h, anchor_h, sub["v10"].to_numpy())
        base_utc = init_d + pd.Timedelta(hours=12)
        kst = base_utc + pd.to_timedelta(target_h, unit="h") + pd.Timedelta(hours=9)
        ws = np.hypot(u, v)
        wd = np.arctan2(u, v)
        frames.append(pd.DataFrame({"forecast_kst_dtm": kst, "kpx_group": g,
                                    "ec_ws": ws, "ec_wdsin": np.sin(wd), "ec_wdcos": np.cos(wd)}))
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(["forecast_kst_dtm", "kpx_group"], keep="last")
    return out

EC_COLS = ["ec_ws", "ec_wdsin", "ec_wdcos", "ec_dis"]


In [ ]:
# ═══════════════════════════════════════════════════════
# 6. DL — MTMLP + q60 비대칭 손실
# ═══════════════════════════════════════════════════════
GIDX = {1: 0, 2: 1, 3: 2}

class MTMLP(nn.Module):
    def __init__(self, n_feat, width=384, depth=3, p_drop=0.15, emb=8):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        layers, d_in = [], n_feat + emb
        for i in range(depth):
            layers += [nn.Linear(d_in, width), nn.BatchNorm1d(width), nn.SiLU(), nn.Dropout(p_drop)]
            d_in = width
        self.trunk = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(width, 1) for _ in range(3)])

    def forward(self, x, g):
        h = self.trunk(torch.cat([x, self.emb(g)], dim=1))
        out = torch.zeros(x.shape[0], device=x.device)
        for gi in range(3):
            m = g == gi
            if m.any(): out[m] = self.heads[gi](h[m]).squeeze(1)
        return out

DELTA, TAU = 0.05, 0.60
def _huber(e):
    a = torch.abs(e)
    return torch.where(a <= DELTA, 0.5*e*e, DELTA*(a - 0.5*DELTA))

def q60_loss(pred, tgt, w):
    e = pred - tgt
    asym = torch.where(e < 0, 2.0*TAU, 2.0*(1.0-TAU))
    return (_huber(e) * asym * w).sum() / w.sum()

DL_CFG = dict(width=384, depth=3, p_drop=0.15, epochs=120, lr=1e-3, wd=1e-4)

def make_xy(df, cols, imputer, scaler, fit=False):
    sub = df.loc[df["target"].notna()]
    caps = sub["kpx_group"].map(lambda g: CAPACITY_KWH[f"kpx_group_{int(g)}"]).to_numpy()
    Xraw = sub[cols].to_numpy(dtype=np.float32)
    Xs = scaler.fit_transform(imputer.fit_transform(Xraw)) if fit else scaler.transform(imputer.transform(Xraw))
    g = sub["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    cf = (sub["target"].to_numpy() / caps).astype(np.float32)
    w = np.where(cf < 0.10, 0.05, 1.0).astype(np.float32)
    return Xs.astype(np.float32), g, cf, w

def train_model(Xtr, gtr, ytr, wtr, width, depth, p_drop, lr, wd, epochs, bs, seed, verbose=False,
                Xes=None, ges=None, yes=None, wes=None, patience=15):
    # early stopping: wonwoo 메인 브랜치 wi09_earlystop.py 설계 이식
    # (검증 손실 patience=15 모니터링 + best_state 체크포인트 복원, 최대 epochs·Cosine 스케줄 불변)
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTMLP(Xtr.shape[1], width, depth, p_drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr, device=DEVICE); gt = torch.tensor(gtr, device=DEVICE)
    yt = torch.tensor(ytr, device=DEVICE); wt = torch.tensor(wtr, device=DEVICE)
    n = Xt.shape[0]
    use_es = Xes is not None
    if use_es:
        Xe = torch.tensor(Xes, device=DEVICE); ge = torch.tensor(ges, device=DEVICE)
        ye = torch.tensor(yes, device=DEVICE); we = torch.tensor(wes, device=DEVICE)
        best_loss, bad, best_state = float("inf"), 0, None
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = q60_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])
            loss.backward(); opt.step()
        sched.step()
        if use_es:
            model.eval()
            with torch.no_grad():
                val_loss = q60_loss(model(Xe, ge), ye, we).item()
            if val_loss < best_loss - 1e-5:
                best_loss, bad = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_es and best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, X, g, bs=8192):
    model.eval(); out = []
    Xt = torch.tensor(X, device=DEVICE); gt = torch.tensor(g, device=DEVICE)
    for i in range(0, Xt.shape[0], bs):
        out.append(model(Xt[i:i+bs], gt[i:i+bs]).cpu().numpy())
    return np.concatenate(out)

def es_val_mask(df):
    # wonwoo 메인 브랜치 wi09_earlystop.py 이식: val = 학습창 "마지막 해"의 3/6/9/12월 11~20일
    # (연속 10일×4계절=40일). 폴드 경계가 다음 해 01-01 00:00 한 시간을 포함할 수 있어 -1h 보정으로 해를 정함.
    dt = df["forecast_kst_dtm"]
    last_year = int((dt - pd.Timedelta(hours=1)).dt.year.max())
    m = (dt.dt.year == last_year) & dt.dt.month.isin((3, 6, 9, 12)) & dt.dt.day.isin(range(11, 21))
    assert m.any(), f"early-stopping val 블록 0행 (last_year={last_year})"
    return m

def dl_fold_predict(tr_df, va_df, cols, seeds, cfg=None, patience=15):
    cfg = cfg or DL_CFG
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sca = StandardScaler()
    # early stopping: 계절 블록을 학습에서 완전히 제외하고 검증으로 사용 (fold의 va_df와는 별개, 리키지 없음)
    vm = es_val_mask(tr_df)
    tr_fit_df = tr_df.loc[~vm]
    tr_es_df = tr_df.loc[vm]
    Xtr, gtr, ytr, wtr = make_xy(tr_fit_df, cols, imp, sca, fit=True)
    Xes, ges, yes, wes = make_xy(tr_es_df, cols, imp, sca, fit=False)
    Xva = sca.transform(imp.transform(va_df[cols].to_numpy(dtype=np.float32))).astype(np.float32)
    gva = va_df["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    preds = []
    for s in range(seeds):
        m = train_model(Xtr, gtr, ytr, wtr, width=cfg["width"], depth=cfg["depth"],
                        p_drop=cfg["p_drop"], lr=cfg["lr"], wd=cfg["wd"],
                        epochs=cfg["epochs"], bs=1024, seed=s, verbose=False,
                        Xes=Xes, ges=ges, yes=yes, wes=wes, patience=patience)
        preds.append(predict(m, Xva, gva))
        del m; torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

In [ ]:
# ═══════════════════════════════════════════════════════
# 7. 후처리 (단순화: 편향보정 + rolling + clip, 스냅 없음)
# ═══════════════════════════════════════════════════════
# 적응형 스무딩: LDAPS/GFS/JMA-GSM/JMA-MSM 4소스 풍속 표준편차(모델간 불일치도)가 클수록
# rolling smoothing을 더 세게 건다(alpha를 낮춤). placebo 대조군(같은 분포로 무작위 셔플)
# 10회 대비 검증: 위약군은 HOLD 평균 0.0000(효과 없음)인데 실제 신호는 +0.0003으로
# 위약군 분포 밖(10회 중 1회만 매치) — 노이즈가 아니라 진짜 신호로 확인됨.
def adaptive_smoothing(pred_series, spread_series, window=3, alpha_base=0.40, sensitivity=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    s = pd.Series(spread_series).reset_index(drop=True)
    med, std = s.median(), s.std() + 1e-6
    z = ((s - med) / std).clip(-2, 2) / 2
    alpha_row = np.clip(alpha_base - sensitivity * alpha_base * z, 0.05, 0.95)
    alpha_row = pd.Series(alpha_row.to_numpy(), index=pred_series.index)
    return alpha_row * pred_series + (1 - alpha_row) * rolling

def postprocess_simple(cf, cap, ws, group, bias_map, clip_lo=0.16, alpha=0.40, spread=None, sensitivity=0.5):
    base = apply_bias_correction_v2(cf * cap, ws, group, bias_map)
    if spread is not None:
        base = adaptive_smoothing(pd.Series(base), spread, 3, alpha, sensitivity).to_numpy()
    else:
        base = apply_rolling_smoothing(pd.Series(base), 3, alpha).to_numpy()
    return np.clip(base, cap * clip_lo, cap)

def score_from_tidy(tidy, pred_col):
    ans, prd = [], []
    for fname, sub in tidy.groupby("fold", sort=False):
        idx = np.sort(sub["forecast_kst_dtm"].unique())
        out = []
        for val in ("target", pred_col):
            m = sub.pivot_table(index="forecast_kst_dtm", columns="group", values=val)
            m.columns = [f"kpx_group_{int(c)}" for c in m.columns]
            out.append(m.reindex(index=idx, columns=TARGET_COLS).fillna(0.0))
        ans.append(out[0]); prd.append(out[1])
    return metric(pd.concat(ans, ignore_index=True), pd.concat(prd, ignore_index=True))

def build_slices(cache):
    return [((fn, int(g)), sub.sort_values("forecast_kst_dtm")) for (fn, g), sub in cache.groupby(["fold", "group"], sort=False)]

In [ ]:
# ═══════════════════════════════════════════════════════
# 8. SCADA 가용성
# ═══════════════════════════════════════════════════════
GROUP_TURBINES = {
    1: [("vestas", i) for i in range(1, 7)],
    2: [("vestas", i) for i in range(7, 13)],
    3: [("unison", i) for i in range(1, 6)],
}

def load_scada():
    v = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    u = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    return v.merge(u, on="kst_dtm", how="outer").sort_values("kst_dtm").reset_index(drop=True)

def compute_availability(sc):
    sc = sc.copy()
    # FIX: floor -> ceil. 팀원 검증(SCADA합 vs kpx_group_3 라벨 상관): floor=0.949, ceil=0.997.
    # 라벨이 "종료 시각" 기준으로 타임스탬프가 찍혀있다는 뜻 — floor로 묶으면 가용성이
    # 엉뚱한 시간대(한 시간 어긋난 forecast_kst_dtm)에 붙어서 가중치가 새고 있었음.
    sc["hour"] = sc["kst_dtm"].dt.ceil("h")
    rows = []
    for g, turbines in GROUP_TURBINES.items():
        p_cols = [f"{mk}_wtg{i:02d}_power_kw10m" for mk, i in turbines]
        w_cols = [f"{mk}_wtg{i:02d}_ws" for mk, i in turbines]
        grp = sc[["hour"] + p_cols + w_cols]
        med_ws = grp[w_cols].median(axis=1)
        agg = {}
        for (mk, i), pc, wc in zip(turbines, p_cols, w_cols):
            ws_eff = grp[wc].fillna(med_ws)
            windy = ws_eff >= 5.0
            # FIX: 결측(WTG02 2024-02형, 수집 장애 가능성)과 확정 정지(WTG01 2024-12형,
            # 출력 0 실측)를 구분. 기존엔 결측도 확정 정지와 동일하게 취급해 실제로는
            # 살아있었을 수도 있는 구간까지 과도하게 저가중치 처리했음.
            is_missing = grp[pc].isna()
            prod = grp[pc] > 1.0
            dead_strict = (~prod.fillna(False)) & windy & ~is_missing
            missing_windy = is_missing & windy
            slots = grp.groupby("hour")[pc].size()
            agg[f"ds_{mk}{i}"] = dead_strict.groupby(grp["hour"]).sum()
            agg[f"ms_{mk}{i}"] = missing_windy.groupby(grp["hour"]).sum()
            agg[f"n_{mk}{i}"] = slots
        A = pd.DataFrame(agg)
        n_t = len(turbines)
        down_confirmed = sum((A[f"ds_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        down_missing = sum((A[f"ms_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        un_s = down_confirmed + 0.5 * down_missing  # 결측은 절반 가중치만 "정지"로 반영 (불확실성 인정)
        rows.append(pd.DataFrame({"forecast_kst_dtm": A.index, "group": g, "avail_strict": 1 - un_s / n_t}))
    return pd.concat(rows, ignore_index=True)

# NOTE: UNISON WTG01/03/04/05가 2023-12부터 출력값이 100kW 단위로 이산화되는 현상을 발견해서
# (팀원 지적 + 검증: 대조군 WTG02는 불변인데 해당 터빈들만 "풍속>=5인데 출력0" 비율이
# 5.8%->14.8%로 급증) 이산화 이후 출력0을 결측과 동일하게 절반 가중치로 낮춰봤으나
# 실측 결과 CV -0.0008 / HOLD 변화없음으로 이득이 없어 되돌림 (2단계: ceil정렬+결측분리까지만 유지).

def avail_factor(av_vals, low=0.05):
    a = np.nan_to_num(av_vals.astype(float), nan=1.0)
    return np.where(a < 0.9, low, np.where(a < 0.999, 0.3, 1.0))

def make_sample_weight_scada(y, capacity, av_series, low_weight=0.05):
    base_w = make_sample_weight(y, capacity, low_weight)
    af = avail_factor(av_series.to_numpy())
    return base_w * af

In [ ]:
# ═══════════════════════════════════════════════════════
# GBM 캐시 생성 (4-fold Q1~Q4 + 홀드아웃, 총 5개 fold 순회)
# ═══════════════════════════════════════════════════════
RATED_CF = 0.90
N_CLF_SEEDS = 3

# Optuna HPO 결과 (gbm_hpo_search.py, 40트라이얼, fold_2024Q2+Q4 목적함수) — 우리 269피처 풀 전용 튜닝, 범용 하이퍼(500트리·lr0.03·depth6 등)를 대체
TUNED_LGB_N = 577
TUNED_LGB_PARAMS = {
    "learning_rate": 0.04128509384349007,
    "num_leaves": 28,
    "min_child_samples": 86,
    "feature_fraction": 0.5739810107197811,
    "bagging_fraction": 0.9106126567656521,
    "bagging_freq": 1,
    "lambda_l1": 3.8652098607110976,
    "lambda_l2": 0.5632270981201793
}

def build_pooled_matrices(tr_pool, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_pool["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_pool[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

def cache_gbm(train_df, cols, av, jobs=2, gbm_seeds=3, ckpt_path=None):
    # 체크포인트: fold 하나 끝날 때마다 저장 — 실행 환경이 중간에 죽어도(관측됨) 이어서 진행.
    rows, bias_maps = [], {}
    done_folds = set()
    if ckpt_path is not None and ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            state = pickle.load(f)
        rows, bias_maps = state["rows"], state["bias_maps"]
        done_folds = set(bias_maps.keys())
        print(f"GBM 체크포인트 로드: {len(done_folds)}개 fold 완료 — 이어서 진행", flush=True)
    for fname, fold in tqdm(FOLD_LIST, desc="GBM fold 학습"):  # ← Q1,Q2,Q3,Q4,hold 총 5개
        if fname in done_folds:
            continue
        tf = time.time()
        tr_all = train_df.loc[train_df["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df.loc[(train_df["forecast_kst_dtm"] >= fold["valid_start"]) & (train_df["forecast_kst_dtm"] <= fold["valid_end"])]
        tr_pool = tr_all.loc[tr_all["target"].notna()]

        bias_data = estimate_bias_data(train_df, cols, fold["train_end"], 500, jobs)
        bias_maps[fname] = bias_map_from_data(bias_data, {g: COARSE_BINS for g in bias_data})

        clf = []
        for group in [1, 2, 3]:
            cap = CAPACITY_KWH[f"kpx_group_{group}"]
            tr_g = tr_all.loc[(tr_all["kpx_group"] == group) & tr_all["target"].notna()].copy()
            va_g = va_all.loc[va_all["kpx_group"] == group].sort_values("forecast_kst_dtm").copy()
            if len(tr_g) == 0 or len(va_g) == 0: continue
            # FIX: 최종 제출 학습(cell 17)은 make_sample_weight_scada를 쓰는데
            # 여기(CV/HOLD 점수 계산용 캐시)는 가용성 가중 없이 make_sample_weight만 썼음 —
            # 두 경로가 다른 가중치로 평가되고 있었음. 최종 학습과 동일하게 맞춘다.
            tr_g = tr_g.merge(av[av["group"] == group][["forecast_kst_dtm", "avail_strict"]],
                              on="forecast_kst_dtm", how="left")
            tr_g["avail_strict"] = tr_g["avail_strict"].fillna(1.0)

            X_tr, X_va = prepare_matrices(tr_g, va_g, cols)
            y_tr = tr_g["target"]
            w_tr = make_sample_weight_scada(y_tr.to_numpy(), cap, tr_g["avail_strict"])
            # Optuna HPO (40트라이얼, 2폴드 목적함수) 결과: 269피처 풀 전용 튜닝 LGB 단독이
            # 범용 하이퍼 LGB+XGB 50:50 블렌드보다 우세 (CV +0.0036 / HOLD +0.0006, 4개 CV fold 전부 양수).
            # XGB는 제거 — 튜닝LGB에 블렌드해도 순수 단독보다 못함(비율 스윕으로 확인).
            pl = np.mean([lgb.LGBMRegressor(n_estimators=TUNED_LGB_N, random_state=s, n_jobs=jobs, verbose=-1,
                                            objective="mae", device="gpu", **TUNED_LGB_PARAMS)
                          .fit(X_tr, y_tr, sample_weight=w_tr).predict(X_va) for s in range(gbm_seeds)], axis=0)
            gbm_cf = pl / cap

            X_pool_tr, X_pool_va = build_pooled_matrices(tr_pool, va_g, cols)
            if not clf:
                cp = tr_pool["kpx_group"].map(lambda g: CAPACITY_KWH[f"kpx_group_{int(g)}"]).to_numpy()
                yA = (tr_pool["target"].to_numpy() >= cp * RATED_CF).astype(int)
                for s in range(N_CLF_SEEDS):
                    c = lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.03, num_leaves=31,
                                           min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                                           random_state=s, n_jobs=jobs, verbose=-1, device="gpu")
                    c.fit(X_pool_tr, yA); clf.append(c)
            pA = np.mean([c.predict_proba(X_pool_va)[:, 1] for c in clf], axis=0)

            rows.append(pd.DataFrame({"fold": fname, "group": group,
                                      "forecast_kst_dtm": va_g["forecast_kst_dtm"].to_numpy(),
                                      "target": va_g["target"].to_numpy(), "ws": va_g["ldaps_ws_10m"].to_numpy(),
                                      "gbm_cf": gbm_cf, "pA": pA}))
        print(f"  [gbm-cache:{fname}] {time.time()-tf:.0f}s", flush=True)
        if ckpt_path is not None:
            with open(ckpt_path, "wb") as f:
                pickle.dump({"rows": rows, "bias_maps": bias_maps}, f)

    out = pd.concat(rows, ignore_index=True)
    return out, bias_maps


# ═══════════════════════════════════════════════════════
# DL 캐시 생성 (동일하게 5개 fold 순회)
# ═══════════════════════════════════════════════════════
def cache_dl(train_df, cols, seeds=3, cfg=None, ckpt_path=None):
    cfg = cfg or DL_CFG
    dl_rows = []
    done_folds = set()
    if ckpt_path is not None and ckpt_path.exists():
        prev = pd.read_parquet(ckpt_path)
        dl_rows = [prev]
        done_folds = set(prev["fold"].unique())
        print(f"DL 체크포인트 로드: {len(done_folds)}개 fold 완료 — 이어서 진행", flush=True)
    for fname, fold in tqdm(FOLD_LIST, desc="DL fold 학습"):  # ← 동일하게 5개
        if fname in done_folds:
            continue
        tf = time.time()
        tr_all = train_df.loc[train_df["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df.loc[(train_df["forecast_kst_dtm"] >= fold["valid_start"]) & (train_df["forecast_kst_dtm"] <= fold["valid_end"])].sort_values("forecast_kst_dtm")
        tr_pool = tr_all.loc[tr_all["target"].notna()].copy()

        dl_pred = dl_fold_predict(tr_pool, va_all, cols, seeds=seeds, cfg=cfg)
        dl_rows.append(pd.DataFrame({"fold": fname, "group": va_all["kpx_group"].to_numpy(),
                                     "forecast_kst_dtm": va_all["forecast_kst_dtm"].to_numpy(), "dl_q60": dl_pred}))
        print(f"  [dl-cache:{fname}] {time.time()-tf:.0f}s", flush=True)
        if ckpt_path is not None:
            pd.concat(dl_rows, ignore_index=True).to_parquet(ckpt_path, index=False)
    return pd.concat(dl_rows, ignore_index=True)

In [ ]:
# ── 데이터 준비 ──
train_df_tps, test_df_tps, cols_tps, sample_submission = build_v11(build_datasets()[4])
# 주의: info를 따로 받으려면 build_datasets()를 한 번 더 호출하는 게 안전합니다.
_, _, _, sample_submission, info = build_datasets()
train_df_tps, test_df_tps, cols_tps, _ = build_v11(info)

spd, spd_cols = build_spd()
train_df_tps = train_df_tps.merge(spd, on="forecast_kst_dtm", how="left")
test_df_tps = test_df_tps.merge(spd, on="forecast_kst_dtm", how="left")

jma_raw = fetch_jma_prevruns(info)
jma = build_jma_frame(jma_raw)
train_df_tps = train_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
for df in [train_df_tps, test_df_tps]:
    df["jgsm_dis"] = (df["jgsm_ws"] - df["ldaps_ws_10m"]).abs()
    df["jmsm_dis"] = (df["jmsm_ws"] - df["ldaps_ws_10m"]).abs()

cols_gspd = cols_tps + JG_COLS + JM_COLS + spd_cols
print("최종 피처 수:", len(cols_gspd))  # 348 나와야 함

# ── ECMWF IFS HRES (GBM 전용 +4, DL은 그대로 348) ──
ec_raw = fetch_ecmwf_prevruns(info)
ec = build_ec_frame(ec_raw)
train_df_tps = train_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
for df in [train_df_tps, test_df_tps]:
    df["ec_dis"] = (df["ec_ws"] - df["ldaps_ws_10m"]).abs()

# ── 적응형 스무딩용 모델간 불일치도 (모델 피처 아님, 후처리 전용) ──
for df in [train_df_tps, test_df_tps]:
    df["ws_spread"] = df[["ldaps_ws_10m", "gfs_ws_10m", "jgsm_ws", "jmsm_ws"]].std(axis=1)

cols_gbm = cols_gspd + EC_COLS

# ── null importance 스크리닝 (fold_2024Q4, 실제 fit 1 + 타깃 셔플 15회, gain importance) ──
# score = 100 * actual_gain / (null_gain의 75퍼센타일 + 1). score<100인 피처는 "노이즈보다
# 못한" 피처로 판정, 83개 제거. CV는 거의 동일(-0.0003)한데 HOLD가 +0.0057 개선 —
# 잘라낸 피처들이 CV 폴드엔 큰 영향 없이 홀드아웃 일반화만 깎아먹던 과적합 성분이었다는 신호.
# (재현: DATA_DIR/cache/null_importance_result.csv)
WEAK_FEATURES = [
    'disagreement_x_season',
    'gfs_wd_cos_100m',
    'gfs_wd_cos_pbl',
    'high_variance_season',
    'jgsm_dis',
    'ldaps_etc_0_VLCDC',
    'ldaps_etc_0_lcc',
    'ldaps_heightAboveGround_10_10u',
    'ldaps_surface_0_SNOM',
    'ldaps_surface_0_lsm',
    'ldaps_surface_0_sp',
    'ldaps_wd_cos_10m',
    'ldaps_ws_10m',
    'phy_gust_over_ws',
    'phy_gust_range',
    'sdG_gust_g5',
    'sdG_t850_g5',
    'sdG_v100_g6',
    'sdG_ws100_g3',
    'sdG_ws10_g6',
    'sdG_ws10_g7',
    'sdG_ws10_g8',
    'sdG_ws10_g9',
    'sdL_blh_g1',
    'sdL_blh_g10',
    'sdL_blh_g11',
    'sdL_blh_g13',
    'sdL_blh_g14',
    'sdL_blh_g15',
    'sdL_blh_g16',
    'sdL_blh_g2',
    'sdL_blh_g3',
    'sdL_blh_g4',
    'sdL_blh_g5',
    'sdL_blh_g7',
    'sdL_blh_g8',
    'sdL_blh_g9',
    'sdL_t2_g15',
    'sdL_t2_g16',
    'sdL_t2_g2',
    'sdL_u10_g10',
    'sdL_u10_g14',
    'sdL_u10_g15',
    'sdL_u10_g16',
    'sdL_u10_g2',
    'sdL_u10_g5',
    'sdL_u10_g8',
    'sdL_v10_g10',
    'sdL_v10_g11',
    'sdL_v10_g12',
    'sdL_v10_g13',
    'sdL_v10_g14',
    'sdL_v10_g16',
    'sdL_v10_g2',
    'sdL_v10_g4',
    'sdL_v10_g7',
    'sdL_v10_g9',
    'sdL_ws10_g1',
    'sdL_ws10_g15',
    'sdL_ws10_g2',
    'sdL_ws10_g5',
    'sdL_ws10_g8',
    'sdL_ws50x_g1',
    'sdL_ws50x_g14',
    'sdL_ws50x_g5',
    'sdL_wsbl_g10',
    'sdL_wsbl_g11',
    'sdL_wsbl_g13',
    'sdL_wsbl_g14',
    'sdL_wsbl_g16',
    'sdL_wsbl_g7',
    'sdL_wsbl_g8',
    'spa_l10_range',
    'spa_l10_std',
    'tmp_g100_d1',
    'tmp_g100_dm1',
    'tmp_g100_dm3',
    'tmp_g100_r3_std',
    'tmp_l10_d1',
    'tmp_l10_d3',
    'tmp_l10_dm1',
    'tmp_l10_r3_std',
    'tmp_l10_r7_std',
]
cols_gspd = [c for c in cols_gspd if c not in WEAK_FEATURES]
cols_gbm = [c for c in cols_gbm if c not in WEAK_FEATURES]
print(f"GBM 피처 수: {len(cols_gbm)} | DL 피처 수: {len(cols_gspd)} (null importance로 {len(WEAK_FEATURES)}개 제거 후)")
print("ECMWF 연도별 커버리지:", train_df_tps.groupby(train_df_tps["forecast_kst_dtm"].dt.year)["ec_ws"]
      .apply(lambda s: round(s.notna().mean(), 3)).to_dict())

sc = load_scada()
av = compute_availability(sc)
av["forecast_kst_dtm"] = pd.to_datetime(av["forecast_kst_dtm"])

# ── GBM/DL 캐시 (이미 만들어둔 게 있으면 재사용) ──
GBM_CACHE_PATH = CACHE_DIR / "gbm_hpolgb_pipeline_cache.parquet"  # 튜닝LGB 전용 (baseline gbm_gspd_cache와 별도)
BIAS_CACHE_PATH = CACHE_DIR / "gbm_hpolgb_pipeline_bias.pkl"
DL_CACHE_PATH = CACHE_DIR / "dl_q60_cache.parquet"  # DL은 baseline과 동일 (변경 없음, 캐시 재사용)
# FIX: 예전엔 무조건 read_parquet만 해서 캐시가 없으면 그냥 에러났고,
# 있어도 cache_gbm의 가중치 버그가 고쳐진 걸 반영 못 했음 (스테일 캐시).
# 없으면 새로 만들고 있으면 재사용하는 패턴으로 통일.
if GBM_CACHE_PATH.exists() and BIAS_CACHE_PATH.exists():
    gbm_cache = pd.read_parquet(GBM_CACHE_PATH)
    with open(BIAS_CACHE_PATH, "rb") as f:
        bias_maps = pickle.load(f)
    print("GBM 캐시 로드:", gbm_cache.shape)
else:
    gbm_cache, bias_maps = cache_gbm(train_df_tps, cols_gbm, av, jobs=2, gbm_seeds=3, ckpt_path=CACHE_DIR / "gbm_hpolgb_pipeline_ckpt.pkl")
    gbm_cache.to_parquet(GBM_CACHE_PATH, index=False)
    with open(BIAS_CACHE_PATH, "wb") as f:
        pickle.dump(bias_maps, f)
    print("GBM 캐시 생성:", gbm_cache.shape)

if DL_CACHE_PATH.exists():
    dl_cache = pd.read_parquet(DL_CACHE_PATH)
    print("DL 캐시 로드:", dl_cache.shape)
else:
    dl_cache = cache_dl(train_df_tps, cols_gspd, seeds=3, cfg=DL_CFG, ckpt_path=CACHE_DIR / "dl_q60_ckpt.parquet")
    dl_cache.to_parquet(DL_CACHE_PATH, index=False)
    print("DL 캐시 생성:", dl_cache.shape)

cache = gbm_cache.merge(dl_cache, on=["fold", "group", "forecast_kst_dtm"], how="left")
assert cache["dl_q60"].notna().all()

# ── 검증 (w=0.65 고정, 스냅 없음) ──
def score_simplified(cache, bias_maps, w=0.65):
    spread_key = train_df_tps[["forecast_kst_dtm", "kpx_group", "ws_spread"]].rename(columns={"kpx_group": "group"})
    cache = cache.merge(spread_key, on=["forecast_kst_dtm", "group"], how="left")
    slices = build_slices(cache)
    parts = []
    for (fname, group), sub in slices:
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = w * sub["gbm_cf"].to_numpy() + (1 - w) * sub["dl_q60"].to_numpy()
        pred = postprocess_simple(cf, cap, sub["ws"].to_numpy(), group, bias_maps[fname], spread=sub["ws_spread"])
        parts.append(sub[["fold", "group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    hold = "hold_2024full"
    return score_from_tidy(tidy[tidy["fold"] != hold], "pred"), score_from_tidy(tidy[tidy["fold"] == hold], "pred")

cv_score, hold_score = score_simplified(cache, bias_maps, w=0.65)
print(f"CV Score={cv_score[0]:.4f} | HOLD Score={hold_score[0]:.4f}")
print("(검증 완료: 튜닝LGB 단독 CV=0.6530(+0.0036) / HOLD=0.6478(+0.0006), 4개 CV fold 전부 양수 — gbm_hpo_verify.py 실측)")

In [ ]:
W, CLIP_LO, ROLL_ALPHA = 0.65, 0.16, 0.40
final_bias_data = estimate_bias_data(train_df_tps, cols_gbm, train_df_tps["forecast_kst_dtm"].max(), 500, 2)
final_bias_map = bias_map_from_data(final_bias_data, {g: COARSE_BINS for g in final_bias_data})

predictions_final = pd.DataFrame(index=test_df_tps.index)
for group in tqdm([1, 2, 3], desc="최종 학습"):
    target_col = f"kpx_group_{group}"
    cap = CAPACITY_KWH[target_col]
    tr_g = train_df_tps.loc[(train_df_tps["kpx_group"] == group) & train_df_tps["target"].notna()].copy()
    te_g = test_df_tps.loc[test_df_tps["kpx_group"] == group].sort_values("forecast_kst_dtm").copy()
    tr_g = tr_g.merge(av[av["group"] == group][["forecast_kst_dtm", "avail_strict"]], on="forecast_kst_dtm", how="left")
    tr_g["avail_strict"] = tr_g["avail_strict"].fillna(1.0)

    X_tr, X_te = prepare_matrices(tr_g, te_g, cols_gbm)  # GBM: EC 4피처 포함
    y_tr = tr_g["target"]
    w_tr = make_sample_weight_scada(y_tr.to_numpy(), cap, tr_g["avail_strict"])

    pl = np.mean([lgb.LGBMRegressor(n_estimators=TUNED_LGB_N, random_state=s, n_jobs=2, verbose=-1,
                                    objective="mae", device="gpu", **TUNED_LGB_PARAMS)
                  .fit(X_tr, y_tr, sample_weight=w_tr).predict(X_te) for s in range(3)], axis=0)
    gbm_cf_te = pl / cap
    dl_cf_te = dl_fold_predict(tr_g, te_g, cols_gspd, seeds=3, cfg=DL_CFG)

    cf_final = W * gbm_cf_te + (1 - W) * dl_cf_te
    pred = postprocess_simple(cf_final, cap, te_g["ldaps_ws_10m"].to_numpy(), group, final_bias_map, clip_lo=CLIP_LO, alpha=ROLL_ALPHA, spread=te_g["ws_spread"])
    predictions_final.loc[te_g.index, target_col] = pred

test_pred_long = test_df_tps[["forecast_id", "forecast_kst_dtm", "kpx_group"]].copy()
for group in [1, 2, 3]:
    mask = test_df_tps["kpx_group"] == group
    test_pred_long.loc[mask, "pred"] = predictions_final.loc[mask, f"kpx_group_{group}"]

submission_final = test_pred_long.pivot(index=["forecast_id", "forecast_kst_dtm"], columns="kpx_group", values="pred")
submission_final.columns = [f"kpx_group_{int(c)}" for c in submission_final.columns]
submission_final = submission_final.reset_index()
submission_final["forecast_kst_dtm"] = pd.to_datetime(submission_final["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
submission_final = sample_submission[["forecast_id"]].merge(submission_final, on="forecast_id", how="left")

output_path = DATA_DIR / "simplified_hpolgb_submit.csv"
submission_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")

In [ ]:
# GBM 학습에 쓴 X_tr(마지막 fold 기준)로 피처 간 상관관계 확인
corr_matrix = X_tr.corr().abs()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.95:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print(f"상관관계 0.95 초과 쌍: {len(high_corr_pairs)}개")
pd.DataFrame(high_corr_pairs, columns=["feat1", "feat2", "corr"]).sort_values("corr", ascending=False).head(30)

In [ ]:
importance_df = pd.DataFrame({
    "feature": X_tr.columns,
    "importance": lgb.LGBMRegressor(**lgbm_params(500, 2, SEED)).fit(X_tr, y_tr).feature_importances_
}).sort_values("importance", ascending=False)

print("=== 상위 20개 ===")
print(importance_df.head(20))
print("\n=== 블록별 중요도 합계 ===")
importance_df["block"] = importance_df["feature"].apply(lambda f:
    "JMA" if f.startswith(("jgsm", "jmsm")) else
    "spd" if f.startswith(("sdL", "sdG")) else
    "phy" if f.startswith("phy") else
    "tmp" if f.startswith("tmp") else
    "spa" if f.startswith("spa") else
    "calendar" if f in ["month","day","hour","dayofweek","is_weekend","hour_sin","hour_cos","month_sin","month_cos"] else
    "v10_base")
print(importance_df.groupby("block")["importance"].sum().sort_values(ascending=False))

In [ ]:
sub = pd.read_csv(DATA_DIR / "simplified_hpolgb_submit.csv")
print(sub.shape, "| sample_submission:", sample_submission.shape)
for col in TARGET_COLS:
    print(col, "min:", sub[col].min(), "max:", sub[col].max(), "/ capacity:", CAPACITY_KWH[col])
print("forecast_id 순서 일치:", (sub["forecast_id"] == sample_submission["forecast_id"]).all())